# Chapter 03: Inputs and Outputs

Pandas can read from and write to a wide variety of file formats and data sources. Knowing the key parameters for each format saves significant time when working with real-world data.

In this notebook we will cover:

- CSV: `read_csv()` and `to_csv()`
- Excel: `read_excel()` and `to_excel()`
- HTML: `read_html()`
- Parquet: `read_parquet()` and `to_parquet()`
- SQL: `read_sql()` basics

We will use the data files in the `data/` directory where possible.

In [ ]:
import pandas as pd
import numpy as np

## CSV Files

CSV (Comma-Separated Values) is the most common format for tabular data. `read_csv()` is one of the most feature-rich functions in pandas.

In [ ]:
# Basic read
df = pd.read_csv('data/example.csv')
df

In [ ]:
# Common parameters
tips = pd.read_csv('data/tips.csv')
tips.head(3)

### Key `read_csv()` Parameters

| Parameter | Description | Default |
|-----------|-------------|--------|
| `sep` | Delimiter character | `','` |
| `header` | Row number for column names | `0` (first row) |
| `names` | List of column names to use | `None` |
| `index_col` | Column to use as index | `None` |
| `usecols` | Subset of columns to read | `None` (all) |
| `dtype` | Data types for columns | inferred |
| `parse_dates` | Columns to parse as dates | `False` |
| `na_values` | Additional strings to treat as NaN | `None` |
| `nrows` | Number of rows to read | `None` (all) |
| `skiprows` | Rows to skip at the start | `None` |
| `encoding` | File encoding | `'utf-8'` |

In [ ]:
# Read only specific columns
pd.read_csv('data/tips.csv', usecols=['total_bill', 'tip', 'day']).head()

In [ ]:
# Read only the first 5 rows (useful for inspecting large files)
pd.read_csv('data/tips.csv', nrows=5)

In [ ]:
# Parse dates during read
sales = pd.read_csv('data/RetailSales_BeerWineLiquor.csv', parse_dates=['DATE'])
print(sales.dtypes)
sales.head()

In [ ]:
# Set a column as the index during read
sales = pd.read_csv('data/RetailSales_BeerWineLiquor.csv',
                     parse_dates=['DATE'],
                     index_col='DATE')
sales.head()

### Writing to CSV with `to_csv()`

In [ ]:
# Write a DataFrame to CSV
subset = tips[['total_bill', 'tip', 'day', 'time']].head(10)
subset.to_csv('data/tips_subset.csv', index=False)

# Verify by reading it back
pd.read_csv('data/tips_subset.csv')

In [ ]:
# Key to_csv parameters:
# index=False  -> do not write the index
# sep='\t'     -> use tab delimiter
# columns=[..] -> write only specific columns
# na_rep='NA'  -> string representation of NaN

## Excel Files

Pandas reads and writes Excel files using the `openpyxl` engine (for `.xlsx` format).

> **Note**: You need `openpyxl` installed: `pip install openpyxl`

In [ ]:
# Read an Excel file
excel_df = pd.read_excel('data/example.xlsx')
excel_df

In [ ]:
# Key read_excel parameters:
# sheet_name=0        -> first sheet (default), or 'Sheet1', or list of names
# sheet_name=None     -> read ALL sheets into a dict of DataFrames
# header=0            -> row for column names
# usecols='A:C'       -> Excel-style column range
# skiprows=2           -> skip first 2 rows

In [ ]:
# Write to Excel
subset.to_excel('data/tips_subset.xlsx', index=False, sheet_name='Tips')

# Verify
pd.read_excel('data/tips_subset.xlsx')

## HTML Tables

`read_html()` reads all `<table>` elements from an HTML file (or URL) and returns a **list** of DataFrames.

> **Note**: Requires `lxml` or `html5lib` installed.

In [ ]:
# Read tables from a local HTML file
tables = pd.read_html('data/simple.html')
print(f"Found {len(tables)} table(s)")
tables[0]

In [ ]:
# read_html can also read from URLs:
# tables = pd.read_html('https://en.wikipedia.org/wiki/List_of_countries_by_GDP')
# This is extremely useful for quick data scraping

## Parquet Files

Parquet is a columnar storage format that is:
- **Fast** to read/write
- **Compact** (uses compression)
- **Type-preserving** (no need to re-parse dates, categoricals, etc.)

It is the recommended format for storing pandas DataFrames when CSV limitations become an issue.

> **Note**: Requires `pyarrow` or `fastparquet` installed.

In [ ]:
# Write to parquet
tips.to_parquet('data/tips.parquet', index=False)

# Read back
tips_pq = pd.read_parquet('data/tips.parquet')
tips_pq.head()

In [ ]:
# Parquet preserves data types
print(tips_pq.dtypes)

In [ ]:
# Compare file sizes
import os

csv_size = os.path.getsize('data/tips.csv')
pq_size = os.path.getsize('data/tips.parquet')

print(f"CSV size:     {csv_size:,} bytes")
print(f"Parquet size: {pq_size:,} bytes")
print(f"Compression ratio: {csv_size / pq_size:.1f}x")

## SQL Databases

Pandas can read from and write to SQL databases using `sqlalchemy` connections. This is a brief overview — for production use, see the SQLAlchemy documentation.

```python
from sqlalchemy import create_engine

# Create a connection to a SQLite database
engine = create_engine('sqlite:///my_database.db')

# Read a SQL query into a DataFrame
df = pd.read_sql('SELECT * FROM customers WHERE age > 30', engine)

# Read an entire table
df = pd.read_sql_table('customers', engine)

# Write a DataFrame to a SQL table
df.to_sql('customers', engine, if_exists='replace', index=False)
```

| `if_exists` | Behaviour |
|---|---|
| `'fail'` | Raise error if table exists (default) |
| `'replace'` | Drop and recreate the table |
| `'append'` | Insert new rows into existing table |

## Choosing the Right Format

| Format | Best for | Drawbacks |
|--------|----------|----------|
| **CSV** | Universality, human-readable, small data | Slow for large files, no type preservation |
| **Excel** | Sharing with non-technical users | Slow, 1M row limit, requires openpyxl |
| **Parquet** | Large datasets, pipelines, type fidelity | Not human-readable, requires pyarrow |
| **SQL** | Relational data, concurrent access, queries | Requires database setup |

## Clean Up

Remove temporary files created during this notebook.

In [ ]:
import os

for f in ['data/tips_subset.csv', 'data/tips_subset.xlsx', 'data/tips.parquet']:
    if os.path.exists(f):
        os.remove(f)
        print(f"Removed {f}")

## Summary

In this notebook we covered pandas I/O:

- **CSV**: `read_csv()` / `to_csv()` — the most common format; control with `sep`, `usecols`, `parse_dates`, `nrows`.
- **Excel**: `read_excel()` / `to_excel()` — uses `openpyxl`; supports `sheet_name` and Excel-style column ranges.
- **HTML**: `read_html()` — scrapes `<table>` elements from files or URLs; returns a list of DataFrames.
- **Parquet**: `read_parquet()` / `to_parquet()` — fast, compact, type-preserving; ideal for data pipelines.
- **SQL**: `read_sql()` / `to_sql()` — works with SQLAlchemy engines for database integration.

Next up: **Pivot Tables** — reshaping data for multi-dimensional analysis.